Create crosswalks between NextGen catchments and donor basins for all four domains

In [1]:
import geopandas as gpd
import pandas as pd
from pathlib import Path
import glob

In [ ]:
# data directory for regionalization
outdir = Path('/home/yuqiong.liu/work/data/ngen_reg/cwt_ngen_donor')
outdir.mkdir(exist_ok=True, parents=True)

# define domains
domains = {'conus': 'CONUS', 'ak': 'Alaska', 'hi': 'Hawaii', 'prvi': 'Puerto_Rico', 'gl':'Great_Lakes'}

In [ ]:
# loop through domains to build crosswalk files between gage and catchment for calibration
ngage = ncats = 0
for domain1, domain in domains.items():

    # check if crosswalk file already exists, skip
    outfile = Path(outdir,'calib_gage_divide_' + domain1 + '.parquet') 
    if outfile.exists():
        print(f'Crosswalk file already exists: {outfile}. Skip')
        continue

    # gather calibration basins and catchments for the domain
    dir1 = Path('/home/yuqiong.liu/work/data/gpkg_v2.2/', domain).resolve(strict=True)

    # identify all gpkg files
    files = glob.glob(f"{dir1}/*.gpkg") 

    # loop through gpkg files to get divide_ids
    df_cats = pd.DataFrame()
    for f1 in files:

        # get divide_id and a few others attributes
        cats = gpd.read_file(f1, layer="divides")
        cols = ['divide_id', 'toid', 'areasqkm', 'vpuid', 'type']
        cats = cats.reindex(columns=cols, fill_value=float("nan"))

        # get gage id
        cats['gage_id'] = Path(f1).name.replace('gages-','').replace('gauge_','').replace('.gpkg', '')

        # move 'gage' to be the first column
        cats.insert(0, 'gage_id', cats.pop('gage_id'))

        # add to the overall dataframe
        df_cats = pd.concat([df_cats, cats], ignore_index=True, axis=0)

    # save crosswalk to file for use in formulation regionalization later
    df_cats.to_parquet(outfile, index=False)

    print(f'There are {len(files)} calibration gages and {len(df_cats)} catchments in the {domain} domain')

    ngage = ngage + len(files)
    ncats = ncats + len(df_cats)

print(f'\nTotal number of basins: {ngage}')
print(f'Total number of catchments: {ncats}')

There are 1533 calibration gages and 165171 catchments in the CONUS domain
There are 26 calibration gages and 8318 catchments in the Alaska domain
There are 31 calibration gages and 103 catchments in the Hawaii domain
There are 50 calibration gages and 545 catchments in the PuertoRico domain
There are 27 calibration gages and 2257 catchments in the GreatLakes domain

Total number of basins: 1667
Total number of catchments: 176394


In [5]:
# check if there are NWMv4 calibration basins with missing in the crosswalks
df_gages = pd.read_csv("/home/yuqiong.liu/work/data/gages_nwm4_calib_all.csv")
df_gages['domain'] = df_gages['domain'].str.replace(' ','',regex=False)

for domain1, domain in domains.items():

    outfile = Path(outdir,'calib_gage_divide_' + domain1 + '.parquet') 
    df1 = pd.read_parquet(outfile)
    gages1 = df1['gage_id'].to_list()

    gages_nwm4 = df_gages[df_gages['domain'] == domain]['gage_id'].to_list()
    gages_missed = [g1 for g1 in gages_nwm4 if g1 not in gages1]
    gages_extra = [g1 for g1 in gages1 if g1 not in gages_nwm4]
    if gages_extra:
        print(f'The following calibration basins are extra in {domain}: {gages_extra}')

    if gages_missed:
        print(f'The following calibration basins are missing in {domain}: {gages_missed}')

The following calibration basins are missing in Alaska: ['15493000', '15056210']
The following calibration basins are extra in GreatLakes: ['02FD001', '02FD001', '02FD001', '02FD001', '02FD001', '02FD001', '02FD001', '02FD001', '02FD001', '02FD001', '02FD001', '02FD001', '02FD001', '02FD001', '02FD001', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '02BA003', '0

In [6]:
print(df_cats)

      gage_id  divide_id       toid   areasqkm  vpuid     type
0     02FD001  cat-21558  nex-21549  17.928201    NaN  network
1     02FD001  cat-21561  nex-21549   8.547044    NaN  network
2     02FD001  cat-21550  nex-21551  14.248898    NaN  network
3     02FD001  cat-21553  nex-21551  10.940986    NaN  network
4     02FD001  cat-21551  nex-21552  16.357638    NaN  network
...       ...        ...        ...        ...    ...      ...
2252  02ED003  cat-17668  nex-17669   9.952240    NaN  network
2253  02ED003  cat-17671  nex-17672   9.893820    NaN  network
2254  02ED003  cat-17673  nex-17674   9.865926    NaN  network
2255  02ED003  cat-17728  nex-17729  14.820945    NaN  network
2256  02ED003  cat-17782  nex-17783   9.686530    NaN  network

[2257 rows x 6 columns]


In [ ]:
domain1 = 'conus'
outfile = Path(outdir,'calib_gage_divide_' + domain1 + '.parquet')
df1 = pd.read_parquet(outfile)
print(df1)

FileNotFoundError: [Errno 2] No such file or directory: '/home/yuqiong.liu/work/data/ngen_reg/cwt_ngen_donor/cwt_ngen_hlr/calib_gage_divide_conus.parquet'